<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_1_Classification_Basics/18_1_6_ccfraud_with_nested_crosseval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Classification: Part 6
## Credit Card Fraud, Continued — Cost-Weighted Objective & Nested CV
**Author:** Brad Sheese

---

In Notebook 5 we tuned the *decision threshold* of a fixed fraud model. But we never asked whether the model itself was as good as it could be. This notebook closes that gap with three more advanced tools — a cost-weighted training objective, systematic hyperparameter search, and nested cross-validation — and stays honest about how little (or how much) each one actually buys us on this dataset.

### Learning Objectives
By the end of this notebook you will be able to:
1. Explain how a custom XGBoost objective function encodes cost asymmetry differently from `scale_pos_weight`, and describe when each approach is appropriate.
2. Identify the circular-logic flaw that makes a hardcoded-threshold cost scorer unreliable inside `GridSearchCV`, and explain why `average_precision` (PR-AUC) is the correct alternative.
3. Describe how nested cross-validation separates hyperparameter tuning from performance estimation, and interpret the resulting PR-AUC scores.
4. Isolate the contribution of threshold tuning vs. hyperparameter tuning when comparing two models on a held-out test set.
5. Interpret a small performance difference between a baseline and tuned model in terms of diminishing returns from hyperparameter search.

In [2]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import (
    GridSearchCV, StratifiedKFold, cross_val_score, cross_val_predict,
    train_test_split
)
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, classification_report, roc_curve, auc,
    precision_recall_curve, make_scorer
)
import xgboost as xgb

# get the data, create a dataframe
creditcard = fetch_openml(name='creditcard', version=1, as_frame=True)
df = creditcard.frame

# Convert Class from category to int (OpenML loads it as categorical)
df['Class'] = df['Class'].astype(int)


X = df.iloc[:,:-1]
y = df['Class']

# Train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
                                      X,
                                      y,
                                      test_size=0.3,
                                      random_state=42,
                                      stratify=y)

print(f"Original features: {X.shape[1]}")
print(f"Training set size: {X_train.shape[0]}")
print(f"Test set class ratio: {y_test.mean():.1%} bad, {(1-y_test.mean()):.1%} good")

# Define consistent cost values for the entire analysis
COST_FP = 100  # Cost of a false positive (annoying a legitimate customer)
COST_FN = 450  # Cost of a false negative (missing a fraudulent transaction)

print(f"Consistent cost assumptions defined:")
print(f"  False Positive (FP) cost: ${COST_FP}")
print(f"  False Negative (FN) cost: ${COST_FN}")
print(f"  FN is {COST_FN/COST_FP:.1f}x more costly than FP")

Original features: 29
Training set size: 199364
Test set class ratio: 0.2% bad, 99.8% good
Consistent cost assumptions defined:
  False Positive (FP) cost: $100
  False Negative (FN) cost: $450
  FN is 4.5x more costly than FP


In [ ]:
# =====================================================================
# BASELINE MODEL — trained with default settings (no hyperparameter tuning)
# =====================================================================
# This gives us a reference point to compare the tuned model against in the
# evaluation cell below. It also supplies cost_optimal_threshold and y_pred,
# which the evaluation cell requires.
baseline_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    eval_metric='aucpr',
    random_state=42,
)
baseline_model.fit(X_train, y_train)

# Default-threshold predictions used in the cost comparison
y_proba = baseline_model.predict_proba(X_test)[:, 1]
y_pred  = baseline_model.predict(X_test)

# =====================================================================
# COMPUTE COST-OPTIMAL THRESHOLD via Out-of-Fold predictions
# =====================================================================
# Threshold is tuned on training OOF data so the test set stays untouched.
print("Generating OOF probabilities for threshold tuning (may take a moment)...")
oof_train_probs = cross_val_predict(
    baseline_model, X_train, y_train, cv=5, method='predict_proba'
)[:, 1]

candidate_thresholds = np.linspace(0.01, 0.99, 100)
train_costs = []
for t in candidate_thresholds:
    y_train_pred = (oof_train_probs >= t).astype(int)
    fp = ((y_train_pred == 1) & (y_train == 0)).sum()
    fn = ((y_train_pred == 0) & (y_train == 1)).sum()
    train_costs.append(fp * COST_FP + fn * COST_FN)

cost_optimal_threshold = candidate_thresholds[np.argmin(train_costs)]

print(f"Baseline model trained.")
print(f"Cost-optimal threshold (from training OOF): {cost_optimal_threshold:.4f}")

### What NOT to Do: The `total_cost_scorer` Pitfall

The next code cell defines a `total_cost_scorer` function. It looks reasonable — compute dollar costs, return the negative total so GridSearch can maximize it. But it contains a critical flaw.

**The flaw:** `total_cost_scorer` converts probabilities to predictions using a hardcoded threshold (`y_prob >= 0.95`). When GridSearch evaluates each hyperparameter combination, it calls this scorer to rank the results. The problem is that the cost-optimal threshold is not the same for every model — it shifts as hyperparameters change. A model with `max_depth=3` might separate classes best at a threshold of 0.20; a model with `max_depth=7` might need a threshold of 0.05 to minimize cost. By locking in 0.95, you are not comparing models on their actual quality — you are comparing "model A at threshold 0.95" against "model B at threshold 0.95." The search will favor whichever model happens to score well at that specific cutoff, not the one that is fundamentally better at distinguishing fraud from legitimate transactions.

**The correct approach:** Use `average_precision` (PR-AUC) as the GridSearch scoring metric. PR-AUC measures the model's ability to rank fraud above non-fraud across *all* possible thresholds simultaneously — it is threshold-independent. Once GridSearch has selected the best hyperparameters, apply cost-based threshold tuning as a separate post-selection step. This two-step process keeps model selection and threshold selection clean and non-circular.

The `total_cost_scorer` is included below so you can see what the flawed version looks like. The actual GridSearch uses `average_precision`.

### Three Strategies for Class Imbalance — When to Use Each

By this point in the series you have seen three distinct approaches. Here is a direct comparison before we use the most advanced one:

| Approach | When it acts | What it changes | Best for |
|---|---|---|---|
| `scale_pos_weight` | Training | Counts minority samples more in the loss | Simple, fast; good default for moderate imbalance |
| Custom objective | Training | Scales the gradient and hessian by dollar costs per sample | Explicit cost asymmetry when cost ratio ≠ class ratio |
| Threshold tuning | Prediction | Moves the decision boundary after training | Always useful as a final step; the only option when you cannot retrain |

**Why a custom objective here instead of `scale_pos_weight`?**
`scale_pos_weight` applies a multiplier equal to the class ratio — here approximately 499× (99.8% / 0.2%). But our cost ratio is only 4.5× ($450 FN / $100 FP). Using `scale_pos_weight` would over-weight the minority class by two orders of magnitude relative to actual costs.

**Intuition behind the custom objective:** XGBoost trains by computing two quantities at each step: a gradient (which direction to push the prediction) and a hessian (how large a step to take). The `cost_weighted_objective` below scales these by `COST_FN` for fraud samples and `COST_FP` for legitimate samples. XGBoost does not learn a dollar amount — it learns that correcting a missed fraud case deserves a proportionally stronger gradient signal than correcting a false alarm.

> **Runtime note:** The next code cell runs a GridSearch and takes approximately 5 minutes. The nested CV cell further below takes approximately 32 minutes. Both can be skipped on a first read — the key ideas are in the text and the summary output shown in the notebook.

In [ ]:
# this takes ~5 minutes to run

# =====================================================================
# STEP 1: Custom objective — tells XGBoost HOW TO TRAIN
# =====================================================================
def cost_weighted_objective(y_true, y_pred):
    """
    Scales XGBoost's gradient and hessian by dollar costs so the model
    treats missing fraud (FN) as a larger correction signal than a false alarm (FP).
    y_pred here are raw logits (pre-sigmoid), not probabilities.
    """
    cost_fp = COST_FP
    cost_fn = COST_FN
    p = 1.0 / (1.0 + np.exp(-y_pred))   # sigmoid: logit → probability
    grad = cost_fn * (p - 1.0) * y_true + cost_fp * p * (1.0 - y_true)
    hess = (cost_fn * y_true + cost_fp * (1.0 - y_true)) * (p * (1.0 - p))
    return grad, hess


# =====================================================================
# STEP 2: Flawed scorer — ILLUSTRATION ONLY, NOT passed to GridSearchCV.
#         See warning markdown above for why this approach fails.
# =====================================================================
def total_cost_scorer(y_true, y_prob):
    y_pred = (y_prob >= 0.95).astype(int)          # hardcoded threshold — the flaw
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    return -(fp * COST_FP + fn * COST_FN)


# =====================================================================
# STEP 3: GridSearch using the CORRECT scorer: average_precision (PR-AUC)
# =====================================================================
grid = GridSearchCV(
    estimator=xgb.XGBClassifier(
        objective=cost_weighted_objective,
        tree_method='hist',
        enable_categorical=True,
        random_state=42
    ),
    param_grid={
        'max_depth': [3, 5, 7],
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.1]
    },
    scoring='average_precision',   # threshold-independent — the correct choice
    cv=3
)
grid.fit(X_train, y_train)

## Nested Cross-Validation with Grid-Search

The GridSearch above finds the best hyperparameters for the specific training split we happened to use. A different random split could produce meaningfully different results. To estimate how well the *entire tuning pipeline* generalizes — not just a single model on a single split — we use nested cross-validation.

> **Note:** Earlier notebooks used `cross_val_predict` to generate out-of-fold probabilities for threshold tuning. Nested CV is different: instead of generating predictions from a fixed model, it runs a complete `GridSearchCV` inside each outer fold, estimating how stable and reliable the *tuning process itself* is across different data splits.

### Structure
* **Outer loop (`cross_val_score`):** Splits the full dataset into 5 parts. For each part, hands the other 4 to the inner loop, then evaluates on the held-out part.
* **Inner loop (`GridSearchCV`):** Within those 4 parts, runs a 3-fold CV grid search to find the best hyperparameters.
* **Result:** 5 PR-AUC scores from 5 independent holdout folds. Consistent scores → stable tuning process. Scattered scores → instability or dataset too small.
* **Compute cost:** 18 grid combinations × 3 inner folds × 5 outer folds = 270 models trained.

> **⚠ Runtime warning:** The cell below runs nested cross-validation and takes approximately **32 minutes** on typical hardware. It is set to `if True:` so it runs by default. If you are working through the notebook in a time-limited session, change `if True:` to `if False:` before running. The key concepts — what nested CV is and why it matters — are explained in the text above. The output you would see is discussed in the cells that follow.

In [ ]:
# this takes 32 minutes to run, so I'm putting it into a conditional
# if you want to run it flip the False to True

# if False:
if True:

  # 1. Define the OBJECTIVE (Unchanged)
  def cost_weighted_objective(y_true, y_pred):
      # Use the globally defined cost constants
      cost_fp = COST_FP
      cost_fn = COST_FN
      p = 1.0 / (1.0 + np.exp(-y_pred))
      grad = cost_fn * (p - 1.0) * y_true + cost_fp * p * (1.0 - y_true)
      hess = (cost_fn * y_true + cost_fp * (1.0 - y_true)) * (p * (1.0 - p))
      return grad, hess

  # 2. Define the SCORER (Unchanged)
  def total_cost_scorer(y_true, y_prob):
      y_pred = (y_prob >= 0.95).astype(int)
      fp = np.sum((y_pred == 1) & (y_true == 0))
      fn = np.sum((y_pred == 0) & (y_true == 1))
      return -(fp * COST_FP + fn * COST_FN)

  # 3. Setup Nested CV Structure
  # Inner CV: Used to find the best hyperparameters
  inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

  # Outer CV: Used to estimate the true performance (generalization error)
  outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

  # Define the Inner Search
  grid_search = GridSearchCV(
      estimator=xgb.XGBClassifier(
          objective=cost_weighted_objective,
          tree_method='hist',
          enable_categorical=True,
          random_state=42
      ),
      param_grid={
          'max_depth': [3, 5, 7],
          'n_estimators': [100, 200, 300],
          'learning_rate': [0.01, 0.1]
      },
      scoring='average_precision',
      cv=inner_cv
  )

  # 4. Run Nested CV
  # This executes the inner search multiple times (once for each outer fold)
  nested_scores = cross_val_score(grid_search, X, y, cv=outer_cv, scoring='average_precision')

  print(f"Nested CV PR-AUC Scores: {nested_scores}")
  print(f"Mean PR-AUC Score: {nested_scores.mean():.4f}")
  print(f"Standard Deviation: {nested_scores.std():.4f}")


Nested CV PR-AUC Scores: [0.84036927 0.88905046 0.86399462 0.86072712 0.82716138]
Mean PR-AUC Score: 0.8563
Standard Deviation: 0.0212


### Nested CV Results for Discussion


In [ ]:
if not grid_search:
    print('Running this cell requires you to have run the previous cell.')

else:
    # ---------------------------------------------------------------
    # Train the FINAL PRODUCTION MODEL on all available data.
    # This is safe now because nested CV has already provided an honest
    # performance estimate. We no longer need to hold data back for
    # evaluation — using all data gives the production model more signal.
    #
    # IMPORTANT: Do NOT evaluate this model's performance on the same data
    # it was trained on. The nested CV mean PR-AUC is the honest estimate.
    # ---------------------------------------------------------------
    grid_search.fit(X, y)
    print("Best hyperparameters:", grid_search.best_params_)

    final_model = grid_search.best_estimator_

    # Production decision threshold
    # We use cost_optimal_threshold from the setup cell above (derived from
    # baseline model OOF probabilities). In a production pipeline you would
    # ideally recompute this using OOF predictions from the final model on the
    # full training set, but the baseline threshold is a close approximation.
    print(f"\nProduction decision threshold: {cost_optimal_threshold:.4f}")
    print(f"Flag any transaction with P(fraud) >= {cost_optimal_threshold:.4f} for review.")

    # Save for deployment
    final_model.get_booster().save_model("fraud_model_v1.json")
    print("Model saved: fraud_model_v1.json")

### What the Results Tell Us About Diminishing Returns

The evaluation below shows the tuned model saves very little over the baseline. This is an intentional teaching moment, not a disappointing outcome.

**Why the gain is small:**

1. **Default XGBoost hyperparameters are strong starting points.** They are not arbitrary — they represent what works well across a wide range of tabular datasets. On many problems, defaults capture 90–95% of attainable performance. Grid search retrieves the remaining 5–10%.

2. **Our grid was coarse.** With 18 combinations (3 depths × 3 n_estimator counts × 2 learning rates), we covered broad strokes, not fine detail. A denser search over wider ranges would take proportionally longer and might find meaningfully different configurations.

3. **Diminishing returns are real and worth measuring.** In production, compute time and engineering complexity are costs too. Whether marginal cost savings justify a full tuning pipeline depends on the application. Part of being a practitioner is knowing when to stop tuning.

4. **The nested CV result is the main output, not the dollar figure.** Consistent outer-fold PR-AUC scores confirm the tuning process is stable. That stability — not a specific dollar amount — is what you report when communicating model reliability.

**Takeaway:** Always quantify what hyperparameter tuning actually bought you. Do not assume it helps; measure it.

In [ ]:
# =====================================================================
# FAIR COMPARISON: THREE ROWS THAT ISOLATE EACH IMPROVEMENT
# =====================================================================
# Row 1: Baseline at default 0.50     → starting point
# Row 2: Baseline at cost-optimal     → effect of threshold tuning alone
# Row 3: Tuned model at cost-optimal  → effect of hyperparameter tuning on top
#
# Note: cost_optimal_threshold was derived from baseline OOF probabilities.
# Applying it to the tuned model is a reasonable approximation; to be fully
# rigorous you would re-run OOF threshold tuning on the tuned model separately.
# =====================================================================

best_model = grid.best_estimator_
y_proba_tuned = best_model.predict_proba(X_test)[:, 1]

def eval_at_threshold(y_true, y_proba, threshold, label):
    y_p = (y_proba >= threshold).astype(int)
    tp  = int(((y_p == 1) & (y_true == 1)).sum())
    fp  = int(((y_p == 1) & (y_true == 0)).sum())
    fn  = int(((y_p == 0) & (y_true == 1)).sum())
    recall_val = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    prec_val   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    cost       = fp * COST_FP + fn * COST_FN
    return {
        'Scenario':       label,
        'Threshold':      f'{threshold:.4f}',
        'Recall':         f'{recall_val:.3f}',
        'Precision':      f'{prec_val:.3f}',
        'Total Cost ($)': f'${cost:,.0f}',
        '_cost':          cost,
    }

rows = [
    eval_at_threshold(y_test, y_proba,       0.50,                   'Baseline @ default (0.50)'),
    eval_at_threshold(y_test, y_proba,       cost_optimal_threshold, f'Baseline @ cost-optimal ({cost_optimal_threshold:.2f})'),
    eval_at_threshold(y_test, y_proba_tuned, cost_optimal_threshold, f'Tuned model @ cost-optimal ({cost_optimal_threshold:.2f})'),
]

display_rows = [{k: v for k, v in r.items() if k != '_cost'} for r in rows]
print("=== Model Comparison on Test Set ===")
display(pd.DataFrame(display_rows))

c0, c1, c2 = rows[0]['_cost'], rows[1]['_cost'], rows[2]['_cost']
print(f"\nEffect of threshold tuning alone (row 1 → row 2):  ${c0 - c1:+,.0f}")
print(f"Effect of hyperparameter tuning  (row 2 → row 3):  ${c1 - c2:+,.0f}")
print(f"Total improvement over baseline  (row 1 → row 3):  ${c0 - c2:+,.0f}")

## Summary and Key Takeaways

This notebook extended the fraud detection pipeline from notebook 5 with three advanced techniques: a cost-weighted custom objective, systematic hyperparameter tuning, and nested cross-validation.

**1. Custom XGBoost Objective vs. `scale_pos_weight`**
The `cost_weighted_objective` modifies the gradient and hessian during training so XGBoost penalizes missed fraud more strongly than false alarms. This is appropriate when cost asymmetry ($450 / $100 = 4.5×) differs substantially from class imbalance (99.8% / 0.2% = 499×). Using `scale_pos_weight = 499` would massively over-correct relative to actual costs; the custom objective lets us specify the exact ratio directly in the training signal.

**2. GridSearch Scorer Design**
The correct GridSearch scorer is `average_precision` (PR-AUC), not a cost function with a hardcoded threshold. A hardcoded threshold creates circular logic — the search favors models that score well at *that specific cutoff*, not models with the best underlying class separation. The correct two-step process: select hyperparameters with a threshold-independent metric, then apply cost-based threshold tuning separately afterward.

**3. Nested Cross-Validation**
Nested CV separates hyperparameter tuning from performance estimation. Each outer fold evaluates the complete tuning pipeline on holdout data it has never influenced. Consistent outer-fold PR-AUC scores signal a stable process. The nested CV mean is the honest generalization estimate — more reliable than any single train/test split.

**4. Isolating Each Improvement**
The three-row comparison table separated the effect of threshold tuning from the effect of hyperparameter tuning. Both were small on this dataset — the baseline XGBoost model was already near its performance ceiling. Diminishing returns from grid search are common; always measure the actual improvement rather than assuming tuning helped.

**5. Production Deployment**
The final model is trained on all available data using the best hyperparameters from GridSearch. Performance is estimated from nested CV, not by evaluating the final model on its own training data. The production threshold is derived from cost-optimal OOF tuning, applied at inference time.